# Лабораторная работа  
## Обучение модели обнаружения объектов с YOLO26 на собственных данных

**Основа:** туториал Roboflow *How to Train a YOLO26 Object Detection Model with Custom Data*  
**Источник:** https://blog.roboflow.com/how-to-train-yolo26-custom-data/

---

### Цель работы
Освоить полный цикл обучения модели обнаружения объектов на собственном датасете:
- самостоятельно собрать и разметить изображения в **Roboflow**;
- экспортировать датасет в формате, совместимом с YOLO;
- обучить модель **YOLO26**;
- сравнить качество и скорость обучения для разных **весов модели**;
- исследовать влияние **гиперпараметров** обучения;
- выполнить инференс и проанализировать ошибки модели.

### Краткая опора на исходный туториал
В статье Roboflow показан базовый пайплайн:
1. включить **GPU** в Colab;
2. установить пакеты `ultralytics`, `supervision`, `roboflow`;
3. скачать датасет через Roboflow API;
4. обучить модель командой вида  
   `yolo task=detect mode=train model=yolo26m.pt data=... epochs=20 imgsz=640 plots=True`;
5. использовать файл `best.pt` для инференса и визуализации результатов.

В данной лабораторной работе этот базовый сценарий расширяется:
- датасет размечается **самостоятельно**;
- проводится сравнение **нескольких предобученных весов**;
- проводится серия экспериментов с **параметрами обучения**.

## 1. Требования к результату

По завершении работы студент должен предоставить:
1. заполненный данный notebook;
2. ссылку на проект в **Roboflow**;
3. обученные веса лучшей модели;
4. краткий отчет с таблицей экспериментов и выводами;
5. примеры удачных и неудачных детекций.

## 2. Исходные условия и ограничения

### Требования к датасету
Выберите **одну прикладную задачу обнаружения объектов**. Примеры:
- кружки / бутылки / телефоны;
- автомобили / велосипеды / самокаты;
- лица / каски / рюкзаки;
- книги / клавиатуры / мыши;
- детали оборудования или любые другие понятные объекты.

### Минимальные требования к данным
- не менее **3 классов** объектов;
- не менее **120 изображений**;
- не менее **40 изображений на класс** суммарно;
- данные должны быть **размечены самостоятельно** в Roboflow;
- желательно обеспечить разнообразие:
  - ракурсов;
  - масштаба;
  - освещения;
  - частичных перекрытий;
  - фона.

### Разделение выборки
Рекомендуемое разбиение:
- train: **70%**
- valid: **20%**
- test: **10%**

## 3. Задание

### Часть A. Подготовка данных
1. Соберите собственный набор изображений для задачи обнаружения объектов.
2. Загрузите изображения в **Roboflow**.
3. Выполните **ручную разметку bounding boxes** для всех объектов.
4. Проверьте качество разметки:
   - нет ли пропущенных объектов;
   - не выходят ли боксы далеко за границы объекта;
   - нет ли перепутанных классов.
5. Сгенерируйте версию датасета и экспортируйте ее в формат, совместимый с **YOLO**.

### Часть B. Базовое обучение
1. Подготовьте окружение.
2. Подключите датасет из Roboflow.
3. Обучите базовую модель YOLO26.
4. Зафиксируйте значения основных метрик.

### Часть C. Сравнение весов
Обучите минимум **3 варианта модели** с разными предобученными весами.  
Например:
- `yolo26n.pt`
- `yolo26s.pt`
- `yolo26m.pt`

Для каждого варианта сравните:
- mAP50;
- mAP50-95;
- precision;
- recall;
- время обучения;
- размер итоговых весов.

### Часть D. Исследование параметров
Проведите не менее **3 дополнительных экспериментов** с параметрами обучения.  
Например, исследуйте влияние:
- `epochs`;
- `imgsz`;
- `batch`;
- `patience`;
- `optimizer`;
- `lr0`.

### Часть E. Анализ результатов
1. Выберите лучшую модель.
2. Выполните инференс на тестовых изображениях.
3. Покажите:
   - 3 примера удачных предсказаний;
   - 3 примера ошибок.
4. Проанализируйте причины ошибок.

In [1]:
!pip install ultralytics -qqq

In [4]:
import os;
if 'KAGGLE_USERNAME' not in os.environ or 'KAGGLE_KEY' not in os.environ:
    raise EnvironmentError("Системные переменные KAGGLE_USERNAME или KAGGLE_KEY не найдены. Перезапусти ядро Jupyter, если setx был выполнен при запущенном сервере.");
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA не обнаружена. Обучение на CPU недопустимо для YOLO.");

import os, torch, kaggle, yaml;
kaggle.api.dataset_download_files('kvdep1/chess-dataset', path='d', unzip=True);

c = ['bishop_black', 'bishop_white', 'king_black', 'king_white', 'knight_black', 'knight_white', 'pawn_black', 'pawn_white', 'queen_black', 'queen_white', 'rook_black', 'rook_white'];
b = os.path.abspath('d/datasets/images');
y = {'train': os.path.join(b, 'train'), 'val': os.path.join(b, 'test'), 'nc': 12, 'names': c};

yp = os.path.abspath('d/c.yaml');
with open(yp, 'w') as f:
    yaml.dump(y, f);

Dataset URL: https://www.kaggle.com/datasets/kvdep1/chess-dataset


In [ ]:
import time, pandas as pd;
from ultralytics import YOLO;
from IPython.display import display;

ws = ['yolo26n.pt', 'yolo26s.pt', 'yolo26m.pt'];
rc = [];
yp = os.path.abspath('d/c.yaml');

for w in ws:
    t0 = time.time();
    m = YOLO(w);
    r = m.train(data=yp, epochs=10, imgsz=640, device=0, project='p', name=w.replace('.pt', ''), amp=False, workers=0);
    t1 = time.time();
    
    wp = f"p/{w.replace('.pt', '')}/weights/best.pt";
    sz = os.path.getsize(wp)/(1024*1024) if os.path.exists(wp) else 0;
    d = r.results_dict;
    
    rc.append({
        'Model': w,
        'mAP50': d.get('metrics/mAP50(B)', 0),
        'mAP50-95': d.get('metrics/mAP50-95(B)', 0),
        'P': d.get('metrics/precision(B)', 0),
        'R': d.get('metrics/recall(B)', 0),
        'Time': t1-t0,
        'Size_MB': sz
    });

df_c = pd.DataFrame(rc);
display(df_c.style.background_gradient(cmap='RdYlGn', subset=['mAP50', 'mAP50-95', 'P', 'R']).background_gradient(cmap='RdYlGn_r', subset=['Time', 'Size_MB']));

Ultralytics 8.4.40  Python-3.11.9 torch-2.7.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\jack_\MAIN\\NLP\d\c.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, over

In [ ]:
import itertools;

eps = [10, 20];
bss = [16, 32];
lrs = [0.01, 0.001];
rd = [];
yp = os.path.abspath('d/c.yaml');

for e, b, l in list(itertools.product(eps, bss, lrs)):
    n = f"y26n_e{e}_b{b}_lr{l}";
    
    t0 = time.time();
    m = YOLO('yolo26n.pt');
    r = m.train(data=yp, epochs=e, batch=b, lr0=l, imgsz=640, device=0, project='p_h', name=n, amp=False, workers=0);
    dt = time.time() - t0;
    
    d = r.results_dict;
    rd.append({
        'E': e, 'B': b, 'LR': l,
        'mAP50': d.get('metrics/mAP50(B)', 0),
        'mAP50-95': d.get('metrics/mAP50-95(B)', 0),
        'Time': dt
    });

df_d = pd.DataFrame(rd);
display(df_d.style.background_gradient(cmap='RdYlGn', subset=['mAP50', 'mAP50-95']).background_gradient(cmap='RdYlGn_r', subset=['Time']));

In [ ]:
import glob, matplotlib.pyplot as plt, matplotlib.image as mpimg;

bm = df_d.loc[df_d['mAP50-95'].idxmax()];
bn = f"y26n_e{int(bm['E'])}_b{int(bm['B'])}_lr{bm['LR']}";
bp = f"p_h/{bn}/weights/best.pt";

m_b = YOLO(bp);
ti = glob.glob('d/datasets/images/test/*.jpg');

t0 = time.time();
pi = m_b.predict(source=ti, save=True, device=0, project='p_inf', name='res');
dt_i = time.time() - t0;

ok, err = [], [];
for p in pi:
    lp = p.path.replace('images', 'labels').replace('.jpg', '.txt');
    gt = len(open(lp).readlines()) if os.path.exists(lp) else 0;
    dt = len(p.boxes);
    (ok if gt == dt else err).append(p.save_dir + '/' + os.path.basename(p.path));

def show(ps, t):
    if not ps: return;
    plt.figure(figsize=(15, 5));
    plt.suptitle(t);
    for i, path in enumerate(ps[:3]):
        plt.subplot(1, 3, i+1);
        plt.imshow(mpimg.imread(path));
        plt.axis('off');
    plt.show();

show(ok, "Success (GT == DT)");
show(err, "Errors (GT != DT)");

## 4. Контрольные вопросы перед выполнением
Ответьте кратко.

1. Что такое задача **object detection** и чем она отличается от **classification**?
2. Что означает **bounding box**?
3. Для чего нужен файл `data.yaml`?
4. Что такое **предобученные веса**?
5. Почему качество разметки критично для качества модели?
6. Что показывают метрики **precision**, **recall**, **mAP50**, **mAP50-95**?

In [ ]:
# Запишите здесь краткие ответы на контрольные вопросы.

## 5. Подготовка среды
Согласно туториалу Roboflow, для запуска используются библиотеки:
- `ultralytics`
- `supervision`
- `roboflow`

Также в статье рекомендуется использовать **Google Colab** с включенным **GPU**.

In [ ]:
# При необходимости выполните установку зависимостей
!pip install "ultralytics>=8.4.0" supervision roboflow -q

In [ ]:
# Проверка окружения
import os
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())

## 6. Подключение датасета из Roboflow

Ниже вставьте код подключения **своего** проекта из Roboflow.  
В статье используется загрузка через API-ключ и вызов `version.download(...)`.

In [ ]:
# Пример-шаблон. Замените значения на свои.
# В Colab удобно хранить ключ в Secrets как ROBOFLOW_API_KEY.

from roboflow import Roboflow

ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace("YOUR_WORKSPACE")
project = workspace.project("YOUR_PROJECT")
version = project.version(YOUR_VERSION_NUMBER)

# Возможный формат экспорта — тот, который поддерживает ваш сценарий YOLO
dataset = version.download("yolov11")

print("Dataset location:", dataset.location)

### Задание 6.1
Укажите параметры вашего датасета:
- тема датасета;
- список классов;
- число изображений;
- число объектов по каждому классу;
- какое разбиение train/valid/test использовано.

In [ ]:
dataset_description = {
    "Тема датасета": "",
    "Классы": [],
    "Число изображений": 0,
    "Число объектов по классам": {},
    "Разбиение": "train/valid/test = "
}

dataset_description

## 7. Базовое обучение модели

В статье в качестве примера используется обучение:
- модель: `yolo26m.pt`
- эпохи: `20`
- размер изображения: `640`

Проведите сначала **базовый запуск**, который затем будет точкой сравнения.

In [ ]:
# Базовое обучение
# При необходимости скорректируйте batch и device под вашу среду.

!yolo task=detect mode=train \
    model=yolo26m.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    plots=True

### Задание 7.1
После завершения обучения:
1. Найдите папку с результатами обучения.
2. Укажите путь к:
   - `best.pt`
   - `last.pt`
   - графикам обучения
3. Запишите основные метрики базового запуска.

In [ ]:
baseline_results = {
    "run_dir": "",
    "best_pt": "",
    "last_pt": "",
    "precision": None,
    "recall": None,
    "mAP50": None,
    "mAP50-95": None,
    "notes": ""
}

baseline_results

## 8. Эксперимент 1 — сравнение разных весов

Обучите минимум три варианта модели с разными начальными весами.

### Рекомендуемые варианты
- `yolo26n.pt`
- `yolo26s.pt`
- `yolo26m.pt`

### Что нужно сравнить
- точность по метрикам;
- время обучения;
- устойчивость обучения;
- размер итоговых весов;
- субъективное качество на тестовых изображениях.

In [ ]:
# Примеры запусков. Выполните минимум 3 обучения с разными весами.

# 1) Nano
!yolo task=detect mode=train \
    model=yolo26n.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    plots=True \
    project=lab_yolo26 \
    name=exp_weights_n

# 2) Small
!yolo task=detect mode=train \
    model=yolo26s.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    plots=True \
    project=lab_yolo26 \
    name=exp_weights_s

# 3) Medium
!yolo task=detect mode=train \
    model=yolo26m.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    plots=True \
    project=lab_yolo26 \
    name=exp_weights_m

### Таблица сравнения весов
Заполните таблицу по результатам эксперимента.

In [ ]:
import pandas as pd

weights_comparison = pd.DataFrame([
    {"model": "yolo26n.pt", "epochs": 20, "imgsz": 640, "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "best_pt_size_mb": None, "notes": ""},
    {"model": "yolo26s.pt", "epochs": 20, "imgsz": 640, "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "best_pt_size_mb": None, "notes": ""},
    {"model": "yolo26m.pt", "epochs": 20, "imgsz": 640, "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "best_pt_size_mb": None, "notes": ""},
])

weights_comparison

### Вывод по эксперименту 1
Сделайте вывод:
- какая модель показала лучшее качество;
- какая модель обучалась быстрее;
- какая модель выглядит наиболее рациональной по соотношению **качество / ресурсы**.

In [ ]:
# Ваш вывод по сравнению весов

## 9. Эксперимент 2 — исследование параметров обучения

Проведите минимум три дополнительных запуска, изменяя параметры.
Ниже приведены возможные направления исследования.

### Варианты параметров
1. **Число эпох**: `20`, `50`, `100`
2. **Размер изображения**: `416`, `640`, `960`
3. **Batch size**: `8`, `16`, `32`
4. **Patience** для early stopping
5. **Начальная скорость обучения (`lr0`)**
6. **Оптимизатор**

In [ ]:
# Примеры экспериментальных запусков.
# Выполните любые 3+ запуска и зафиксируйте результаты.

# Эксперимент A: больше эпох
!yolo task=detect mode=train \
    model=yolo26s.pt \
    data={dataset.location}/data.yaml \
    epochs=50 \
    imgsz=640 \
    plots=True \
    project=lab_yolo26 \
    name=exp_params_epochs50

# Эксперимент B: меньший imgsz
!yolo task=detect mode=train \
    model=yolo26s.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=416 \
    plots=True \
    project=lab_yolo26 \
    name=exp_params_img416

# Эксперимент C: другой batch
!yolo task=detect mode=train \
    model=yolo26s.pt \
    data={dataset.location}/data.yaml \
    epochs=20 \
    imgsz=640 \
    batch=16 \
    plots=True \
    project=lab_yolo26 \
    name=exp_params_batch16

In [ ]:
params_comparison = pd.DataFrame([
    {"experiment": "epochs=50", "model": "yolo26s.pt", "imgsz": 640, "batch": "default", "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "notes": ""},
    {"experiment": "imgsz=416", "model": "yolo26s.pt", "imgsz": 416, "batch": "default", "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "notes": ""},
    {"experiment": "batch=16", "model": "yolo26s.pt", "imgsz": 640, "batch": 16, "precision": None, "recall": None, "mAP50": None, "mAP50-95": None, "train_time": None, "notes": ""},
])

params_comparison

### Вывод по эксперименту 2
Ответьте:
1. Какие параметры сильнее всего повлияли на качество?
2. Какие параметры сильнее всего повлияли на скорость?
3. В каких случаях модель начала переобучаться?
4. Какой набор параметров вы считаете оптимальным для вашего датасета?

In [ ]:
# Ваш вывод по исследованию параметров

## 10. Инференс лучшей модели

Выберите лучшую модель и выполните предсказания на тестовых изображениях.
В исходном туториале для инференса используется загрузка `best.pt` и вызов `model.predict(...)`.

In [ ]:
# Пример инференса лучшей модели
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt
import os

BEST_MODEL_PATH = "PASTE_PATH_TO_BEST_PT"

model = YOLO(BEST_MODEL_PATH)

# Укажите путь к одной или нескольким тестовым картинкам
test_images = [
    # "path/to/test1.jpg",
    # "path/to/test2.jpg",
]

for img_path in test_images:
    result = model.predict(img_path, verbose=False)[0]
    annotated = result.plot()
    plt.figure(figsize=(8, 8))
    plt.imshow(annotated[:, :, ::-1])  # BGR -> RGB
    plt.title(os.path.basename(img_path))
    plt.axis("off")
    plt.show()

### Задание 10.1
Приведите:
- **3 примера удачных детекций**;
- **3 примера неудачных детекций**.

Для каждого неудачного случая укажите причину:
- мало примеров такого объекта в обучении;
- сложный фон;
- маленький размер объекта;
- перекрытие объектов;
- ошибки разметки;
- несбалансированность классов;
- неподходящие параметры обучения.

In [ ]:
successful_cases = [
    {"image": "", "comment": ""},
    {"image": "", "comment": ""},
    {"image": "", "comment": ""},
]

failure_cases = [
    {"image": "", "comment": "", "possible_reason": ""},
    {"image": "", "comment": "", "possible_reason": ""},
    {"image": "", "comment": "", "possible_reason": ""},
]

successful_cases, failure_cases

## 11. Итоговый анализ

Сформулируйте развёрнутый вывод по лабораторной работе.

В выводе отразите:
1. какую задачу обнаружения объектов вы решали;
2. как был собран и размечен датасет;
3. какие веса модели сравнивались;
4. какие параметры изменялись;
5. какая конфигурация оказалась лучшей;
6. какие типичные ошибки делает модель;
7. что можно улучшить в будущем.

In [ ]:
final_conclusion = '''
'''
print(final_conclusion)

## 12. Что должно быть в отчёте

В отчёт необходимо включить:
1. тему и цель работы;
2. описание датасета и классов;
3. скриншоты или ссылку на разметку в Roboflow;
4. таблицу сравнения весов;
5. таблицу сравнения гиперпараметров;
6. итоговые метрики лучшей модели;
7. примеры инференса;
8. анализ ошибок;
9. итоговый вывод.